In [3]:
%reload_ext autoreload
%autoreload 2

In [4]:
import pandas as pd
# get the general sde synthethic data

df = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/gen_data_stage2/instructor/sdev2_test2/gpt-4o-mini")

df.head()

,id,url1,title,text,prob,questions,context,search_terms,quality,request_tokens,response_tokens,total_tokens,time_taken
0,/SDE/CCMC_Website/|https://ccmc.gsfc.nasa.gov/...,https://ccmc.gsfc.nasa.gov/publicData/workshop...,Index of /publicData/workshops/ICCMC-LWS-2017/...,Index of /publicData/workshops/ICCMC-LWS-2017/...,0.999899,[What is the size of the document titled 'Jon_...,[Jon_Vandegriff_getting_started_with_casestudi...,"[ICCMC LWS 2017 discussions, data architecture...",low,2212.0,360.0,2572.0,6.385803
1,/SDE/CCMC_Website/|https://ccmc.gsfc.nasa.gov/...,https://ccmc.gsfc.nasa.gov/publicData/workshop...,Index of /publicData/workshops/2022/presentati...,Index of /publicData/workshops/2022/presentati...,0.999899,[What is the file size of the presentation tit...,[03_Belehaki_PITHIA-N..>\n\n2022-06-09 22:10\n...,"[presentation schedule June 2022, file sizes o...",low,2254.0,488.0,2742.0,11.795665
2,/SDE/CCMC_Website/|https://ccmc.gsfc.nasa.gov/...,https://ccmc.gsfc.nasa.gov/publicData/workshop...,Index of /publicData/workshops/2018/03_Wednesd...,Index of /publicData/workshops/2018/03_Wednesd...,0.999898,[What is the size of the file 3b.propagators.p...,[3b.propagators.pdf\n\n2018-05-02 19:43\n\n386...,"[public data workshops 2018, 3b.propagators.pd...",low,2115.0,486.0,2601.0,8.769288
3,/SDE/CCMC_Website/|https://ccmc.gsfc.nasa.gov/...,https://ccmc.gsfc.nasa.gov/publicData/workshop...,Index of /publicData/workshops/2016/5_Friday_AM,Index of /publicData/workshops/2016/5_Friday_A...,0.999898,[What are the file sizes of the presentations ...,[Name\nLast modified\nSize\nDescription\nParen...,"[2016 Friday AM workshop presentations, presen...",low,2214.0,280.0,2494.0,5.221994
4,/SDE/CCMC_Website/|https://ccmc.gsfc.nasa.gov/...,https://ccmc.gsfc.nasa.gov/publicData/kameleon/,Index of /publicData/kameleon,Index of /publicData/kameleon\n\nIndex of /pub...,0.999898,[What is the size of the CORHEL_MAS.tar.bz2 fi...,"[CORHEL_MAS.tar.bz2\n2022-08-05 15:15\n83M, EN...","[CORHEL_MAS.tar.bz2 file size, ENLIL.tar.bz2 l...",low,2063.0,871.0,2934.0,18.926588


In [8]:
df["quality"].value_counts()

quality
high      9241
medium     608
low        128
Name: count, dtype: int64

In [10]:
# only input datapoint with high quality
df = df[df["quality"] == "high"]
df.shape

(9241, 13)

# QA explosion

In [11]:
import json
# generate QA pairs

qa_pairs = df.explode(['questions', 'context']).reset_index(drop=True)
qa_pairs = qa_pairs.rename(columns={'questions': 'query', 'context': 'context'})
qa_pairs["type"] = "QA Pair"

# Remove NaN values
qa_pairs = qa_pairs.dropna(subset=['query', 'context'])

columns_to_keep = ["query", "context", "type"]
qa_pairs = qa_pairs[columns_to_keep]
qa_pairs.head()

,query,context,type
0,What causes the mysterious clock-like bursts o...,Astronomers propose that the flashes are due t...,QA Pair
1,What is the location of the star-forming regio...,The object LRLL 54361 lies inside the star-for...,QA Pair
2,What does the artist's impression depict regar...,This is an artist's impression of two young bi...,QA Pair
3,Which organizations collaborated on the LRLL 5...,Credit: NASA/ESA/JPL-Caltech/R. Hurt (IPAC) sh...,QA Pair
4,When was the artist's impression of LRLL 54361...,"Artwork • February 7th, 2013 • ssc2013-04b.",QA Pair


In [12]:
qa_pairs.shape

(87150, 3)

# search_term explosion

In [13]:
import numpy as np

sde_search_pairs = df.explode(['search_terms']).reset_index(drop=True)
sde_search_pairs = sde_search_pairs.rename(columns={'search_terms': 'query', 'text': '_context'})

sde_search_pairs.drop(columns=['context'], inplace=True)
sde_search_pairs = sde_search_pairs.rename(columns={'_context': 'context'})
sde_search_pairs["type"] = "Search Pair"

# Remove NaN values
sde_search_pairs = sde_search_pairs.dropna(subset=['query', 'context'])

columns_to_keep = ["query", "context", "type"]
sde_search_pairs = sde_search_pairs[columns_to_keep]
sde_search_pairs.head()

# query	context	metadata

,query,context,type
0,pulsating star LRLL 54361,Artist's Impression of Pulsating Object LRLL 5...,Search Pair
1,circumstellar disk dynamics,Artist's Impression of Pulsating Object LRLL 5...,Search Pair
2,binary stars radiation bursts,Artist's Impression of Pulsating Object LRLL 5...,Search Pair
3,IC 348 star-forming region,Artist's Impression of Pulsating Object LRLL 5...,Search Pair
4,NASA JPL binary protostar research,Artist's Impression of Pulsating Object LRLL 5...,Search Pair


In [14]:
sde_search_pairs.shape

(99933, 3)

# combine  / concat all datasets

In [15]:
df_all = pd.concat([qa_pairs, sde_search_pairs], ignore_index=True)

In [16]:
df_all.head()

,query,context,type
0,What causes the mysterious clock-like bursts o...,Astronomers propose that the flashes are due t...,QA Pair
1,What is the location of the star-forming regio...,The object LRLL 54361 lies inside the star-for...,QA Pair
2,What does the artist's impression depict regar...,This is an artist's impression of two young bi...,QA Pair
3,Which organizations collaborated on the LRLL 5...,Credit: NASA/ESA/JPL-Caltech/R. Hurt (IPAC) sh...,QA Pair
4,When was the artist's impression of LRLL 54361...,"Artwork • February 7th, 2013 • ssc2013-04b.",QA Pair


In [17]:
df_all.shape

(187083, 3)

In [18]:
df_all.drop_duplicates(inplace=True)
df_all.shape

(186688, 3)

# converting it to standard jsonal format

In [19]:
import json
import os
from collections import defaultdict

# Create output directory structure
output_dir = "/rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/benchmark_data"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/qrels", exist_ok=True)

print(f"Creating benchmark data in: {output_dir}")

Creating benchmark data in: /rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/benchmark_data


In [20]:
# 1. Create corpus.jsonl - unique documents with IDs
corpus_data = []
corpus_dict = {}
doc_id = 0

# Get unique contexts
unique_contexts = df_all['context'].dropna().drop_duplicates()

for context in unique_contexts:
    if pd.isna(context):  # Extra safety check
        continue
    corpus_dict[context] = str(doc_id)
    corpus_data.append({
        "_id": str(doc_id),
        "text": context
    })
    doc_id += 1

# Write corpus.jsonl
corpus_path = f"{output_dir}/corpus.jsonl"
with open(corpus_path, 'w', encoding='utf-8') as f:
    for doc in corpus_data:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"Created corpus.jsonl with {len(corpus_data)} documents")

Created corpus.jsonl with 82608 documents


In [21]:
# 2. Create queries.jsonl - unique queries with IDs
queries_data = []
query_dict = {}
query_id = 0

# Get unique queries
unique_queries = df_all['query'].dropna().drop_duplicates()

for query in unique_queries:
    if pd.isna(query):  # Extra safety check
        continue
    query_dict[query] = str(query_id)
    queries_data.append({
        "_id": str(query_id),
        "text": query
    })
    query_id += 1

# Write queries.jsonl
queries_path = f"{output_dir}/queries.jsonl"
with open(queries_path, 'w', encoding='utf-8') as f:
    for query in queries_data:
        f.write(json.dumps(query, ensure_ascii=False) + '\n')

print(f"Created queries.jsonl with {len(queries_data)} queries")

Created queries.jsonl with 176901 queries


In [22]:
# 3. Create qrels TSV files - separate for QA pairs and Search pairs
qa_qrels = []
search_qrels = []

for _, row in df_all.iterrows():
    query_id = query_dict[row['query']]
    doc_id = corpus_dict[row['context']]
    
    # Format: query_id doc_id relevance_score
    qrel_entry = f"{query_id}\t{doc_id}\t1"
    
    if row['type'] == 'QA Pair':
        qa_qrels.append(qrel_entry)
    elif row['type'] == 'Search Pair':
        search_qrels.append(qrel_entry)

# Write QA pairs qrels
qa_qrels_path = f"{output_dir}/qrels/qa_pairs.tsv"
with open(qa_qrels_path, 'w', encoding='utf-8') as f:
    # Write header
    f.write("query-id\tcorpus-id\tscore\n")
    for qrel in qa_qrels:
        f.write(qrel + '\n')

# Write Search pairs qrels
search_qrels_path = f"{output_dir}/qrels/search_pairs.tsv"
with open(search_qrels_path, 'w', encoding='utf-8') as f:
    # Write header
    f.write("query-id\tcorpus-id\tscore\n")
    for qrel in search_qrels:
        f.write(qrel + '\n')

print(f"Created QA pairs qrels with {len(qa_qrels)} entries")
print(f"Created Search pairs qrels with {len(search_qrels)} entries")

Created QA pairs qrels with 86775 entries
Created Search pairs qrels with 99913 entries


In [23]:
# 4. Create summary statistics and verification
print("\n=== BENCHMARK DATASET SUMMARY ===")
print(f"Total unique documents: {len(corpus_data)}")
print(f"Total unique queries: {len(queries_data)}")
print(f"QA pairs: {len(qa_qrels)}")
print(f"Search pairs: {len(search_qrels)}")
print(f"Total query-document pairs: {len(qa_qrels) + len(search_qrels)}")

# Verify data integrity
print(f"\nData integrity check:")
print(f"Original dataframe rows: {len(df_all)}")
print(f"Total qrels entries: {len(qa_qrels) + len(search_qrels)}")
print(f"Match: {len(df_all) == len(qa_qrels) + len(search_qrels)}")

# Show file structure
print(f"\n=== OUTPUT FILES ===")
print(f"📁 {output_dir}/")
print(f"├── corpus.jsonl ({len(corpus_data)} documents)")
print(f"├── queries.jsonl ({len(queries_data)} queries)")
print(f"└── qrels/")
print(f"    ├── qa_pairs.tsv ({len(qa_qrels)} pairs)")
print(f"    └── search_pairs.tsv ({len(search_qrels)} pairs)")


=== BENCHMARK DATASET SUMMARY ===
Total unique documents: 82608
Total unique queries: 176901
QA pairs: 86775
Search pairs: 99913
Total query-document pairs: 186688

Data integrity check:
Original dataframe rows: 186688
Total qrels entries: 186688
Match: True

=== OUTPUT FILES ===
📁 /rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/benchmark_data/
├── corpus.jsonl (82608 documents)
├── queries.jsonl (176901 queries)
└── qrels/
    ├── qa_pairs.tsv (86775 pairs)
    └── search_pairs.tsv (99913 pairs)


In [24]:
# 5. Optional: Create sample files for inspection
print("\n=== SAMPLE DATA ===")

# Sample corpus
print("Sample corpus entry:")
print(json.dumps(corpus_data[0], indent=2, ensure_ascii=False))

# Sample query
print("\nSample query entry:")
print(json.dumps(queries_data[0], indent=2, ensure_ascii=False))

# Sample qrels
print("\nSample QA qrels (first 3 lines):")
with open(qa_qrels_path, 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            print(line.strip())

print("\nSample Search qrels (first 3 lines):")
with open(search_qrels_path, 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            print(line.strip())


=== SAMPLE DATA ===
Sample corpus entry:
{
  "_id": "0",
  "text": "Astronomers propose that the flashes are due to material in a circumstellar disk suddenly being dumped onto the growing young stars and unleashing a blast of radiation each time the stars get close to each other in their orbit."
}

Sample query entry:
{
  "_id": "0",
  "text": "What causes the mysterious clock-like bursts of light from the object LRLL 54361?"
}

Sample QA qrels (first 3 lines):
query-id	corpus-id	score
0	0	1
1	1	1

Sample Search qrels (first 3 lines):
query-id	corpus-id	score
86181	73389	1
86182	73389	1


In [ ]:
# # 6. Optional: Create combined qrels file
# combined_qrels_path = f"{output_dir}/qrels/combined.tsv"
# with open(combined_qrels_path, 'w', encoding='utf-8') as f:
#     f.write("query-id\tcorpus-id\tscore\n")
#     for qrel in qa_qrels + search_qrels:
#         f.write(qrel + '\n')

# print(f"\nAlso created combined qrels file: {combined_qrels_path}")
# print(f"Combined qrels entries: {len(qa_qrels) + len(search_qrels)}")


Also created combined qrels file: /rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/benchmark_data/qrels/combined.tsv
Combined qrels entries: 105939
